# Random forest: pre-bid risk

Run `python preprocessing.py` once from the project folder. The input has one row per auction, and IsBadBuy = 1 is the positive class. Compare the two objectives on the same validation cars using continuous scores.

In [ ]:
import pandas as pd
from preprocessing import TRIM_FIELDS, CATEGORICAL, ROOT, load_development
from preprocessing import NUMERIC, ACQUISITION_PRICES
from evaluation import save_validation

# Selected 15-field configuration; use False only to reproduce the 11-field baseline.
include_prices = True
X_train, y_train, X_val, y_val = load_development(include_prices=include_prices)
cat_fields = CATEGORICAL + TRIM_FIELDS
result_dir = ROOT / "results" / "validation"
if include_prices:
    result_dir = result_dir / "acquisition_prices"
scores = pd.DataFrame(index=y_val.index)
print(f"{len(X_train):,} training cars, {len(X_val):,} validation cars")

### Train-only encoding
Fit one shared encoder for the two forests. An unseen category is ignored by the encoder.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder

numeric_fields = NUMERIC + (ACQUISITION_PRICES if include_prices else [])
encoder = ColumnTransformer([
    ("category", OneHotEncoder(handle_unknown="ignore"), cat_fields),
    ("number", "passthrough", numeric_fields),
], sparse_threshold=1.0)
train_encoded = encoder.fit_transform(X_train)
val_encoded = encoder.transform(X_val)

### Matched forest settings
Each tree uses squared error or log loss to choose splits. Both forests use the same sampling and feature budget.

In [ ]:
settings = dict(n_estimators=160, max_depth=14, min_samples_leaf=12,
                max_features="sqrt", bootstrap=True, random_state=42, n_jobs=2)

rf_l2 = RandomForestRegressor(criterion="squared_error", **settings)
rf_l2.fit(train_encoded, y_train)
scores["RF_L2"] = rf_l2.predict(val_encoded)

rf_log = RandomForestClassifier(criterion="log_loss", **settings)
rf_log.fit(train_encoded, y_train)
scores["RF_Logloss"] = rf_log.predict_proba(val_encoded)[:, 1]

### Validation AUC
Scores and labels are saved by RefId for a paired comparison with CatBoost.

In [ ]:
save_validation(y_val, scores, result_dir, "random_forest")